# Methodology: Workflow-Centric Collaboration Analysis

This notebook provides a step-by-step walkthrough of the workflow-centric methodology used to analyze human-agent collaboration in software development. Each section demonstrates how the analysis is derived directly from the PR timeline data.

## Overview

The methodology analyzes **33,600 complete PR workflows** across 5 AI coding tools by:

1. **Loading raw PR timeline data** from JSON files
2. **Classifying actors** (Agent vs Human), using `actor.type == "Bot"` when present and falling back to naming patterns
3. **Defining workflow phases** using temporal boundaries
4. **Calculating metrics** (weighted collaboration scores, revision cycles)
5. **Classifying collaboration types** using a hierarchical priority system
6. **Validating results** with deterministic aggregate statistics and verified case studies

### Canonical artifacts (avoid drift)

- **Canonical headline numbers** are produced deterministically into `.tmp/evidence_stats.json` and `.tmp/evidence_stats.md` via:
  - `python -m src.analysis.evidence_stats`
- **Layer 4 audit** validates `reports/evidence_report.md` against the deterministic stats:
  - `python -m src.audit.evidence`

All results shown here are derived directly from the data in `data/raw/pr_timelines_*.json` and the canonical analyzer in `src.analysis.workflows`.

## 1. Setup and Data Loading

First, we set up the environment and load the raw PR timeline data.

In [1]:
import json
import sys
from collections import defaultdict, Counter
from dataclasses import dataclass, field
from pathlib import Path
from typing import List, Dict, Optional, Tuple
from datetime import datetime
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Configure encoding for Windows (only in non-notebook environments)
try:
    sys.stdout.reconfigure(encoding='utf-8')
except (AttributeError, TypeError):
    # In Jupyter notebooks, sys.stdout doesn't support reconfigure
    pass

# Set up paths
REPO_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATASET_DIR = REPO_ROOT / "data" / "raw"

# Add repository root to Python path for imports
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print(f"Repository root: {REPO_ROOT}")
print(f"Dataset directory: {DATASET_DIR}")
print(f"Dataset directory exists: {DATASET_DIR.exists()}")

Repository root: d:\OneDrive - University of Toronto\Year 2026\AIWare
Dataset directory: d:\OneDrive - University of Toronto\Year 2026\AIWare\data\raw
Dataset directory exists: True


In [2]:
# Define dataset files
FILES = {
    'Claude': 'pr_timelines_Claude_Code.json',
    'Copilot': 'pr_timelines_Copilot.json',
    'Cursor': 'pr_timelines_Cursor.json',
    'Devin': 'pr_timelines_Devin.json',
    'OpenAI': 'pr_timelines_OpenAI_Codex.json'
}

# Verify files exist
for tool, filename in FILES.items():
    filepath = DATASET_DIR / filename
    if filepath.exists():
        size_mb = filepath.stat().st_size / (1024 * 1024)
        print(f"✓ {tool:10s}: {filename:40s} ({size_mb:.1f} MB)")
    else:
        print(f"✗ {tool:10s}: {filename:40s} (NOT FOUND)")

✓ Claude    : pr_timelines_Claude_Code.json            (18.5 MB)
✓ Copilot   : pr_timelines_Copilot.json                (257.4 MB)
✓ Cursor    : pr_timelines_Cursor.json                 (46.5 MB)
✓ Devin     : pr_timelines_Devin.json                  (208.4 MB)
✓ OpenAI    : pr_timelines_OpenAI_Codex.json           (307.2 MB)


### 1.1 Data Structure

Each dataset file is a JSON object where:
- **Keys**: PR identifiers (e.g., `"3193888615.json"`)
- **Values**: Arrays of timeline event objects

Let's examine the structure of a single PR timeline:

In [3]:
# Load a sample PR from Devin dataset (contains LangBot#1398)
devin_file = DATASET_DIR / FILES['Devin']

# Use streaming parser to avoid loading entire file
from src.analysis.stream_pr_timelines import iter_timeline_items

# Get first PR as example
sample_pr_key = None
sample_events = []

for item in iter_timeline_items(devin_file):
    if item.item is None:
        if sample_pr_key:
            break
        continue
    if sample_pr_key is None:
        sample_pr_key = item.pr_key
    if item.pr_key == sample_pr_key:
        sample_events.append(item.item)

print(f"Sample PR Key: {sample_pr_key}")
print(f"Number of events: {len(sample_events)}")
print(f"\nFirst event structure:")
print(json.dumps(sample_events[0], indent=2)[:500] + "...")

Sample PR Key: 2768448959.json
Number of events: 4

First event structure:
{
  "sha": "4554ec7cf0e300a2930a5b8cff8f583f065aaed7",
  "node_id": "C_kwDOCcqYF9oAKDQ1NTRlYzdjZjBlMzAwYTI5MzBhNWI4Y2ZmOGY1ODNmMDY1YWFlZDc",
  "url": "https://api.github.com/repos/whitphx/vscode-emacs-mcx/git/commits/4554ec7cf0e300a2930a5b8cff8f583f065aaed7",
  "html_url": "https://github.com/whitphx/vscode-emacs-mcx/commit/4554ec7cf0e300a2930a5b8cff8f583f065aaed7",
  "author": {
    "name": "Devin AI",
    "email": "158243242+devin-ai-integration[bot]@users.noreply.github.com",
    "date": "202...


## 2. Actor Classification

The first step is to classify each event's actor as either **Agent** (automation/bot actor) or **Human** (developer).

### 2.1 Bot Identification Patterns

When `actor.type` is present in the dataset, it can provide a direct Bot/User signal. Otherwise, we identify agents using naming patterns observed in the data (fallback heuristic):

In [4]:
# Bot identification patterns (from workflows.py)
BOT_PATTERNS = ['[bot]', 'bot', 'copilot', 'devin', 'cursor', 'codex', 'openai', 'claude']

def is_bot(actor: Optional[dict]) -> bool:
    """Check if an actor looks like a bot/Agent actor (heuristic fallback)."""
    if not actor or not isinstance(actor, dict):
        return False
    login = str(actor.get('login', actor.get('name', ''))).lower()
    return any(p in login for p in BOT_PATTERNS)

def get_actor_name(event: dict) -> str:
    """Extract actor name from event."""
    actor = (event.get('actor') or event.get('author') or 
             event.get('user') or event.get('committer'))
    if actor and isinstance(actor, dict):
        return actor.get('login', actor.get('name', 'unknown'))
    return 'unknown'

def is_agent_event(event: dict) -> bool:
    """Use actor.type when present; otherwise fall back to naming heuristic."""
    actor = (event.get('actor') or event.get('author') or 
             event.get('user') or event.get('committer'))
    actor_type = actor.get('type') if isinstance(actor, dict) else None
    if actor_type == 'Bot':
        return True
    return is_bot(actor)

# Demonstrate actor classification on sample events
print("Actor Classification Examples:")
print("=" * 60)
for i, event in enumerate(sample_events[:10]):
    actor_name = get_actor_name(event)
    is_agent = is_agent_event(event)
    event_type = event.get('event', 'unknown')
    print(f"Event {i+1}: {event_type:20s} | Actor: {actor_name:30s} | {'AGENT' if is_agent else 'HUMAN'}")

Actor Classification Examples:
Event 1: committed            | Actor: Devin AI                       | AGENT
Event 2: commented            | Actor: devin-ai-integration[bot]      | AGENT
Event 3: closed               | Actor: devin-ai-integration[bot]      | AGENT
Event 4: head_ref_deleted     | Actor: whitphx                        | HUMAN


## 3. Workflow Lifecycle Model

The methodology uses a **5-phase PR lifecycle** model based on temporal boundaries rather than event types alone.

### 3.1 Phase Definitions

The phases are:

1. **INITIATION**: PR created → First `committed` event
2. **DEVELOPMENT**: First `committed` → First `review_requested` after commits
3. **REVIEW**: `review_requested` → `approved` OR `changes_requested`
4. **REVISION**: `changes_requested` + subsequent commit → Next `review_requested`
5. **RESOLUTION**: `merged` or `closed` → Terminal

### 3.2 Temporal Boundary Detection

Let's implement the phase detection logic:

In [5]:
@dataclass
class WorkflowEvent:
    """Represents a single event in the PR workflow."""
    index: int
    event_type: str
    actor: str
    is_agent: bool
    timestamp: Optional[str] = None
    
    @property
    def actor_label(self) -> str:
        return "Agent" if self.is_agent else "Human"

@dataclass
class WorkflowPhase:
    """Represents a phase in the workflow lifecycle."""
    name: str
    events: List[WorkflowEvent] = field(default_factory=list)
    start_index: Optional[int] = None
    end_index: Optional[int] = None
    
    @property
    def agent_count(self) -> int:
        return sum(1 for e in self.events if e.is_agent)
    
    @property
    def human_count(self) -> int:
        return sum(1 for e in self.events if not e.is_agent)
    
    @property
    def primary_actor(self) -> str:
        if not self.events:
            return "None"
        if self.agent_count > self.human_count:
            return "Agent"
        elif self.human_count > self.agent_count:
            return "Human"
        return "Mixed"

def parse_timestamp(ts: Optional[str]) -> Optional[datetime]:
    """Parse ISO8601 timestamp."""
    if not ts:
        return None
    try:
        if ts.endswith('Z'):
            ts = ts[:-1] + '+00:00'
        return datetime.fromisoformat(ts)
    except:
        return None

In [6]:
def assign_phases_temporal(events: List[WorkflowEvent]) -> Dict[str, WorkflowPhase]:
    """
    Assign events to phases using temporal boundaries.
    
    This implements the 5-phase model from the methodology:
    1. INITIATION: PR created → First committed
    2. DEVELOPMENT: First committed → First review_requested after commits
    3. REVIEW: review_requested → approved OR changes_requested
    4. REVISION: changes_requested + commit → Next review_requested
    5. RESOLUTION: merged/closed → Terminal
    """
    phases = {
        'initiation': WorkflowPhase('initiation'),
        'development': WorkflowPhase('development'),
        'review': WorkflowPhase('review'),
        'revision': WorkflowPhase('revision'),
        'resolution': WorkflowPhase('resolution')
    }
    
    current_phase = 'initiation'
    first_commit_seen = False
    first_review_request_seen = False
    in_revision_cycle = False
    
    for i, event in enumerate(events):
        event_type = event.event_type
        
        # Phase transitions based on temporal boundaries
        if event_type == 'committed' and not first_commit_seen:
            first_commit_seen = True
            current_phase = 'development'
            phases['initiation'].end_index = i - 1
            phases['development'].start_index = i
        
        elif event_type == 'review_requested' and first_commit_seen and not first_review_request_seen:
            first_review_request_seen = True
            if not in_revision_cycle:
                current_phase = 'review'
                phases['development'].end_index = i - 1
                phases['review'].start_index = i
        
        elif event_type == 'reviewed':
            # Check review state
            review_state = None  # Would extract from event data
            if review_state == 'CHANGES_REQUESTED':
                in_revision_cycle = True
                current_phase = 'revision'
                if phases['review'].end_index is None:
                    phases['review'].end_index = i - 1
                phases['revision'].start_index = i
            elif review_state == 'APPROVED':
                if current_phase == 'review':
                    phases['review'].end_index = i
                current_phase = 'resolution'
                phases['resolution'].start_index = i
        
        elif event_type == 'committed' and in_revision_cycle:
            # Revision commit - stay in revision phase
            current_phase = 'revision'
        
        elif event_type == 'review_requested' and in_revision_cycle:
            # Back to review after revision
            in_revision_cycle = False
            current_phase = 'review'
            if phases['revision'].end_index is None:
                phases['revision'].end_index = i - 1
            phases['review'].start_index = i
        
        elif event_type in ('merged', 'closed'):
            current_phase = 'resolution'
            if phases['resolution'].start_index is None:
                phases['resolution'].start_index = i
        
        # Add event to current phase
        phases[current_phase].events.append(event)
    
    return phases

print("Phase assignment function defined.")
print("This implements the temporal boundary model from the methodology.")

Phase assignment function defined.
This implements the temporal boundary model from the methodology.


## 4. Weighted Collaboration Metrics

The methodology uses **weighted event contributions** to measure collaboration balance, recognizing that not all events have equal significance.

### 4.1 Event Weights

In [7]:
# Event weights from methodology
EVENT_WEIGHTS = {
    'committed': 3.0,  # Core development activity
    'reviewed': {  # Depends on state
        'CHANGES_REQUESTED': 2.0,  # Substantive feedback
        'APPROVED': 1.5,  # Gating decision
        'COMMENTED': 1.0  # Neutral feedback
    },
    'commented': 1.0,  # Discussion contribution
    'review_requested': 0.5,  # Process coordination
    'assigned': 0.25,  # Metadata operations
    'labeled': 0.25,  # Metadata operations
    'default': 0.5  # Default weight for unlisted events
}

def get_event_weight(event: WorkflowEvent, review_state: Optional[str] = None) -> float:
    """Get weight for an event based on type and context."""
    event_type = event.event_type
    
    if event_type in EVENT_WEIGHTS:
        weight = EVENT_WEIGHTS[event_type]
        if isinstance(weight, dict):
            # For reviewed events, use state-specific weight
            return weight.get(review_state, weight.get('COMMENTED', 1.0))
        return weight
    
    return EVENT_WEIGHTS['default']

def calculate_weighted_contribution(events: List[WorkflowEvent], is_agent: bool) -> float:
    """Calculate weighted contribution for an actor type."""
    total = 0.0
    for event in events:
        if event.is_agent == is_agent:
            # In practice, would extract review_state from event data
            weight = get_event_weight(event)
            total += weight
    return total

def calculate_collaboration_score(agent_weighted: float, human_weighted: float) -> float:
    """Calculate weighted collaboration score."""
    if agent_weighted == 0 and human_weighted == 0:
        return 0.0
    return min(agent_weighted, human_weighted) / max(agent_weighted, human_weighted)

print("Event Weights:")
for event_type, weight in EVENT_WEIGHTS.items():
    if not isinstance(weight, dict):
        print(f"  {event_type:20s}: {weight:.2f}")
print("\nWeighted collaboration score formula:")
print("  Score = min(Agent_weighted, Human_weighted) / max(Agent_weighted, Human_weighted)")

Event Weights:
  committed           : 3.00
  commented           : 1.00
  review_requested    : 0.50
  assigned            : 0.25
  labeled             : 0.25
  default             : 0.50

Weighted collaboration score formula:
  Score = min(Agent_weighted, Human_weighted) / max(Agent_weighted, Human_weighted)
